### Persons in the geographical space

We present various methods of spatialising the birthplaces of our population.

The data was prepared using the queries [documented on this page](https://github.com/Sciences-historiques-numeriques/astronomers/blob/main/documentation/wikidata/data-analysis/da2-birth-places.sql)

In [ ]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import LineString
from shapely import wkt

import scipy.stats as stats
import math

from geopandas.tools import sjoin
#from geodatasets import get_path

In [ ]:
import matplotlib.pyplot as plt

import plotly.express as px
import plotly.graph_objects as go


from pathlib import Path


import json

import numpy as np
import seaborn as sns
import math
import os

In [ ]:
import warnings
warnings.filterwarnings('ignore')


In [ ]:

import pprint
import csv


from shutil import copyfile


## Geo data

Documentation: [Geographical Information System](https://en.wikipedia.org/wiki/Geographic_information_system) (Wikipedia)

<img src=./doc_images/DatendimensionenImGIS.jpg alt="geodata" width="50%">  
Par F Lamiot d'après [1] (GNUFDL) — source, CC BY-SA 3.0, https://commons.wikimedia.org/w/index.php?curid=3831140


### Source des données spatiales
[Word Regions (ESRI)](https://hub.arcgis.com/datasets/a79a3e4dc55343b08543b1b6133bfb90/explore?location=-0.027457%2C0.000000%2C0.88). For personal use only, data under copyright !

In [ ]:
world_filepath = 'geo_data/World_Regions_6144914380456424035.geojson'
## dataframe GeoPandas
world = gpd.read_file(world_filepath)
world.head()

### Basic GIS notions

* [Geographic information system - GIS](https://en.wikipedia.org/wiki/Geographic_information_system)
* [Geoid](https://en.wikipedia.org/wiki/Geoid)
* [Earth ellipsoid](https://en.wikipedia.org/wiki/Earth_ellipsoid): WGS84
* [Spatial reference system - SRS/CRS](https://en.wikipedia.org/wiki/Spatial_reference_system)
* [EPSG Geodetic Parameter Dataset](https://en.wikipedia.org/wiki/EPSG_Geodetic_Parameter_Dataset): EPSG:4326, EPSG:3857, EPSG:9989

In [ ]:
### Inspect the projection
pprint.pprint(world.crs)

In [ ]:
### Draw the world map

## GeoPandas documentation
# https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html

# If we want to exclude the Antarctic region
world.clip([-180.0, -10.0, 180.0, 90.0])

ax = world.plot(color="DarkCyan", alpha=0.2, edgecolor="black", figsize=(12,9))

# If we want to exclude the Antarctic region
ax.set_xlim(-170,180)
ax.set_ylim(-65, 90)


plt.show()



## Import the data

The data is available in the da_data directory and was prepared using the query [documented on this page](../../documentation/wikidata/data-analysis/da1-distribution-of-births-in-time.md)

In [ ]:
## create variable declaring the path to the data
path_to_data = 'da_data/da2-birth-place.csv'

In [ ]:
## creating a Pandas Dataframe (data container in form of a table) 
df_p = pd.read_csv(path_to_data, sep=",", encoding="utf-8", header=0)
df_p.columns = ['uriPer', 'labelPer', 'birthYear', 'gender', 'labelPlace',
                'geometry','uriPlace']

## first rows of the dataframe
df_p.head()

In [ ]:
## Basic infor about the DataFrame
df_p.info()

## Inspect the data


In [ ]:
### Add column names and inspect length of dataframe

print('Number: ',len(df_p))
df_p.iloc[10:13,:]

### Treat multiple birth places

Verify that persons are unique:
* are there some persons with multiple birth places, etc.?
  * If Pandas *is_unique* method: 'true' they are unique, so no multiple.
  * If 'false': multiple rows per person
* If you have multiple persons go back to the SQL part and clean up using GROUP BY -> MIN  

In [ ]:
### Tests if the values in the column are unique
# If unique result : True (boolean value not string !)
is_unique=df_p.uriPer.is_unique
print(is_unique)

## Group and map birth places

"Well-known text (WKT) is a text markup language for representing vector geometry objects" ([Wikipedia](https://en.wikipedia.org/wiki/Well-known_text_representation_of_geometry))

In [ ]:
### Group and count the number of persons per birth place
p_gr = df_p.groupby(by=['uriPlace', 'labelPlace', 'geometry'], as_index=False).size()
p_gr.columns=['uriPlace', 'labelPlace', 'geometry', 'size']
p_gr.sort_values('size', ascending=False).head()


In [ ]:
### Try first to convert WKT into Point Geometry
# using a geopandas feature
try:
    p_gr['geometry'] = p_gr['geometry'].apply(wkt.loads)
except Exception as e:
    print(e)

In [ ]:
### there are wrong values in the data: find them
# all the rows where the geometry field does not contain the string 'POINT'
df_p.loc[~df_p.geometry.str.contains('POINT')]

In [ ]:
### Drop the rows with the wrong value
df_p=df_p.loc[df_p.geometry.str.contains('POINT')]

In [ ]:
### Re-do grouping after cleaning up
p_gr = df_p.groupby(by=['uriPlace', 'labelPlace', 'geometry'], as_index=False).size()
p_gr.columns=['uriPlace', 'labelPlace', 'geometry', 'number']
p_gr.sort_values('number', ascending=False).head(2)


In [ ]:

### Create a dataframe with the POINT geometry
# https://geopandas.org/en/stable/gallery/create_geopandas_from_pandas.html
birth_gdf = gpd.GeoDataFrame(
    p_gr, 
    ### Use this if no conversion before to geometry
    ## Cf. above:  p_gr['geometry'].apply(wkt.loads)
    geometry=gpd.GeoSeries.from_wkt(p_gr.geometry), 
    crs=4326
)

birth_gdf.sort_values('number', ascending=False).head(2)



In [ ]:
### The Python class of the object changed 
# WKT strings are now points
print(p_gr.geometry.dtype, '->', birth_gdf.geometry.dtype)

In [ ]:
### Inspect the projection
pprint.pprint(birth_gdf.crs)

In [ ]:
### Draw the world map


ax = world.plot(color="LightCyan", alpha=0.3, 
                edgecolor="black", linewidth=2, figsize=(64,48))

# If we want to exclude the Antarctic region
ax.set_xlim(-170,180)
ax.set_ylim(-60, 80)

# We can now plot our ``GeoDataFrame``.
birth_gdf.plot(ax=ax, color="DarkRed", markersize='number', alpha=0.5)
# Add labels where number > 100
for _, row in birth_gdf[birth_gdf['number'] > 50].iterrows():
    ax.annotate(
        text=row['labelPlace'],          # e.g. "Boston"
        xy=(row.geometry.x, row.geometry.y),
        xytext=(5, 5),              # offset in points
        textcoords="offset points",
        fontsize=7,
        color="Black",
        bbox=dict(
            boxstyle="round,pad=0.2",
            facecolor="white",
            edgecolor="none",
            alpha=0.7
        )
    )

f_address = "images/birth_places_geopandas.png"
plt.savefig(f_address, bbox_inches='tight', pad_inches=0.05, dpi=150)
plt.show()
print('Done -> inspect picture')
plt.close()

## Experiment with different kinds of maps

In [ ]:
birth_gdf.head(1)

In [ ]:
### Create columns for long / lat
birth_gdf['long'] = birth_gdf['geometry'].x
birth_gdf['lat'] = birth_gdf['geometry'].y
birth_gdf.head(1)

In [ ]:
birth_gdf_plus1 = birth_gdf.loc[birth_gdf.number>1]
print(birth_gdf_plus1[["lat", "long"]].head(10))
print(birth_gdf_plus1[["lat", "long"]].describe())
print(birth_gdf_plus1.crs)

In [ ]:
### We explore different ways of normalizing the data,
## i.e. diminishing the difference between high and low values
# p_size = [int(s/10+2) for s in birth_gdf_plus1['number']]
# p_size = [int(s) for s in birth_gdf_plus1['number']]
p_size = [(math.log(s)) for s in birth_gdf_plus1['number']]
max(p_size)

In [ ]:

fig = px.scatter_map(
    birth_gdf_plus1, lat="lat", lon="long", size=p_size,
    hover_name="labelPlace", hover_data=["number"],
    opacity=0.8, zoom=3.5,
    center=dict(lon=8.5, lat=47),
    height=1000
)

fig.update_traces(marker=dict(color="GreenYellow", sizemin=0.2))

fig.update_layout(
    map=dict(
        style={
            "version": 8,
            "sources": {
                "usgs": {
                    "type": "raster",
                    "tiles": ["https://basemap.nationalmap.gov/arcgis/rest/services/USGSImageryOnly/MapServer/tile/{z}/{y}/{x}"],
                    "tileSize": 256
                }
            },
            "layers": [{
                "id": "usgs-layer",
                "type": "raster",
                "source": "usgs"
            }]
        },
        center=dict(lon=8.5, lat=47),
        zoom=3.5
    )
)

#fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
f_address = "interactive_images/birth_places_geological_survey.html"
fig.write_html(f_address)
print('Done -> inspect picture')
# fig.show()
plt.close()

In [ ]:
### Open the world regions ESRI file in geojson format (and not dataframe)
with open('geo_data/World_Regions_6144914380456424035.geojson') as f:
    geojson = json.load(f)

geojson['features'][3]['properties']


In [ ]:
### 

p_size = [int(s/10+2) for s in birth_gdf_plus1['number']]

fig = px.scatter_map(birth_gdf_plus1, lat="lat", lon="long", size=p_size, 
                     hover_name="labelPlace", 
                        color_discrete_sequence=["DodgerBlue"],  zoom=3.5,
                        center=dict(lon=8.5, lat=47),height=600,
                        opacity=0.9)
fig.update_layout(
    map_style="white-bg",
    map_layers=[
        {
            "below": 'traces',
            "sourcetype": "geojson",
            "type":"line",
            "sourceattribution": "ESRI",
            "source": geojson,
            "line": {"width": 0.5},
        }
      ])

f_address = "interactive_images/birth_places_regions.html"
fig.write_html(f_address, include_plotlyjs='cdn')
#fig.show()

### Plot the world as a sphere

In [ ]:
### Plot on world as a sphere

lon = birth_gdf["long"]
lat= birth_gdf["lat"]
label= birth_gdf["labelPlace"]
size=[s/10 + 3 for s in birth_gdf["number"]]


# Create the figure
fig = go.Figure(data=go.Scattergeo(
    lon=lon,
    lat=lat,
    mode='markers+text',
    marker=dict(size=size,
                color='red'),
    hovertext=label,
    textposition='top center',
    hoverinfo='text',
    hoverlabel=dict(
        bgcolor='white',
        font_size=12,
        font_family='Arial'
    )
))

# Set the projection to orthogonal
fig.update_layout(
    width=800, 
    height=800,
    geo=dict(
        projection=dict(
            type='orthographic'
        )
    )
)

f_address = "interactive_images/birth_places_globus.html"
fig.write_html(f_address, include_plotlyjs='cdn')
# Show the plot
#fig.show()

### Join regions and inspect

We use here spatial joins and associate a region to each place

In [ ]:
birth_gdf.loc[birth_gdf.labelPlace.str.contains('Petersburg') |
              birth_gdf.labelPlace.str.contains('Moscow')
              ].sort_values('number', ascending=False).head(10)

In [ ]:
### GeoPandas spatial join

w_birth_gdf= birth_gdf.sjoin(world)
w_birth_gdf.iloc[[1,5,10]]

In [ ]:
### Number of regions
print(len(world))
world.head(1)

In [ ]:
### Count by region (sum number of each place in the region)
swb = w_birth_gdf.groupby(['FID','REGION'], as_index=False)['number'].sum()
#swb = swb.set_index('FID', drop=False)
print(len(swb))
swb.sort_values('number', ascending=False).head(24)


In [ ]:
### Find places in European Russia with most births 
w_birth_gdf.loc[w_birth_gdf.FID==9].sort_values('number',ascending=False).head(2)

In [ ]:
### Normal Pandas join: add polygons to aggregated regions 
geom_swb=pd.merge(world, swb, left_on='FID', right_on='FID', 
                  how='left')

geom_swb.head(3)

In [ ]:
"""grp_geo = gpd.GeoDataFrame(
    geom_reg_per, 
    crs=4326
)
"""
### simplify geometry 
## https://www.statology.org/how-to-simplify-geographic-data-using-geopandas/
geom_swb["geometry"] = (
    geom_swb.simplify(tolerance=0.5)
)

In [ ]:
### range
max = geom_swb.number.max()

# Create the choropleth map
fig = px.choropleth_map(
    geom_swb,
    geojson=geom_swb.__geo_interface__,
    map_style='white-bg',
    locations=geom_swb.index,
    color='number',  # Replace with your actual column name
    color_continuous_scale='Blues',
    range_color=(0, max),  # Replace with your actual range
    zoom=1,
    hover_name='REGION_x',
    center={'lat': 47, 'lon': 8.5},
    height=600,
    width=800

)

f_address = "interactive_images/birth_places_regions_choropleth.html"
fig.write_html(f_address, include_plotlyjs='cdn')
# Show the plot
#fig.show()

## Analyse using periods

In [ ]:
### Imputed activity periods
l_10 = list(range(1870, 2030, 10))
print(l_10[:5],l_10[-5:])

In [ ]:
### convert birthYear to integer
df_p.birthYear = df_p.birthYear.apply(lambda x : int(x))

In [ ]:
try:
    df_p = df_p.drop('len_coord', axis=1)
except Exception as e:
    print(e)
df_p.head(2)

In [ ]:
### fonction pd.cut : https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.cut.html
# On ajoute une nouvelle colonne qui contient la période sur la base de la liste précédente
# et de la valeur de l'année
df_p['periods'] = pd.cut(df_p['birthYear'], l_10, right=False)

### Transformer le code ajouté pour qu'il soit plus lisible
# noter qu'on a arrondi les valeurs
df_p['periods'] = df_p['periods'].apply(lambda x : str(int(x.left))+'-'+ str(int(x.right)-1))

df_p=df_p.sort_values(by='periods')

# Inspection
df_p.iloc[[1,5,10]]

In [ ]:
p_gr_per = df_p.groupby(by=['uriPlace', 'labelPlace', 'periods', 'geometry'], observed=True, as_index=False).size()
p_gr_per.columns=['uriPlace', 'labelPlace', 'period', 'coordinates', 'size']
#p_grp.geometry = gpd.GeoSeries.from_wkt(p_gr["coordinates"])
p_gr_per=p_gr_per.sort_values(by='period')
p_gr_per.head()


In [ ]:

### Create a Geopandas geometry column
# https://geopandas.org/en/stable/gallery/create_geopandas_from_pandas.html
birth_p_grp = gpd.GeoDataFrame(
    p_gr_per, 
    ### Here we use it
    geometry=gpd.GeoSeries.from_wkt(p_gr_per.coordinates), crs=4326
)

birth_p_grp.head(3)



In [ ]:
### Create columns for long / lat
birth_p_grp['long'] = birth_p_grp['geometry'].x
birth_p_grp['lat'] = birth_p_grp['geometry'].y
birth_p_grp.head()

In [ ]:
# now just plot it on a map with evolution by time
# https://plotly.com/python/animations/
# https://plotly.com/python/scatter-plots-on-maps/

# np.log(s) * 1000
size = [s*5 if s != 0 else 0 for s in birth_p_grp["size"]]




fig = px.scatter_geo(
    birth_p_grp,
    lon="long",
    lat="lat",
    size=size,
    hover_name = "labelPlace",
    animation_frame= "period",
    width=1400, height=600,
    color_discrete_sequence=['red'],
    title="Evolution of birth places through time"
)
    
fig.layout.updatemenus[0].buttons[0].args[1]["frame"] = {"duration": 1500, "redraw": True}
fig.layout.updatemenus[0].buttons[0].args[1]["transition"] = {"duration": 800, "easing": "cubic-in-out"}


fig.update_layout(
    mapbox={"style": "carto-positron", "zoom":2},
    margin={"l": 0, "r": 20, "t": 30, "b": 200}
)



fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 1000
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 20

### Enregistrer l'image, puis l'ouvrir dans un autre onglet du navigateur
f_address = "interactive_images/birth_places_periods.html"
fig.write_html(f_address, include_plotlyjs='cdn')
#fig.show()
plt.close()


### Merge persons and territories

This merge provides regions as additional properties of persons

In [ ]:
with open('geo_data/World_Regions_6144914380456424035.geojson') as f:
    geojson = json.load(f)

pprint.pprint(geojson['features'][3]['properties'])

In [ ]:
print([f['properties']['FID'] for f in geojson['features']][:3])

In [ ]:
## dataframe GeoPandas from geojson features
world = gpd.GeoDataFrame.from_features(geojson['features'])
### this is needed with this import procedure
world.set_crs(epsg=4326, inplace=True)
world.head()

In [ ]:
### Inspect the projection
pprint.pprint(world.crs)

In [ ]:
world_filepath = 'geo_data/World_Regions_6144914380456424035.geojson'
## dataframe GeoPandas
world = gpd.read_file(world_filepath)
world.head()

In [ ]:
### Cleaned up dataframe
df_p.iloc[[1,5,10]]

In [ ]:
### create a geo data frame

gdf_p = gpd.GeoDataFrame(
    df_p, 
    ### Here we use it
    geometry=gpd.GeoSeries.from_wkt(df_p.geometry), crs=4326
)

gdf_p.head(3)



In [ ]:
### Number of persons before join
print(len(gdf_p))

In [ ]:
### spatial join : if POINT is inside REGION -> join 
# per person we add the region of birth
pgr_reg = gdf_p.sjoin(world)
pgr_reg.iloc[[1,5,10]]

In [ ]:
### There are somme issues with the spatial join and 
### around 600 persons are 'lost'
print(len(pgr_reg))

In [ ]:
### drop not neede column
pgr_reg = pgr_reg.drop([ 'index_right', 'SQMI','SQKM'], axis=1)
pgr_reg.head(2)

#### Prepare the data for the bivariate analysis, cf. challenge da3

In [ ]:
file_address='da3-birthYear-gender-region.csv'
pgr_reg.to_csv(file_address, index=False)

## Merge persons and countries

In [ ]:
### Source Eurostat: https://ec.europa.eu/eurostat/web/gisco/geodata/administrative-units/countries
## © EuroGeographics for the administrative boundaries
countries_filepath = 'geo_data/CNTR_RG_03M_2024_4326.geojson'
## dataframe GeoPandas
countries = gpd.read_file(countries_filepath)
countries.head()

In [ ]:
countries.info()

In [ ]:
pprint.pprint(countries.crs)

In [ ]:
### Draw the world map with the countries

## GeoPandas documentation
# https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.plot.html

# If we want to exclude the Antarctic region
#countries.clip([-180.0, -10.0, 180.0, 90.0])

ax = countries.plot(color="DarkCyan", alpha=0.2, edgecolor="black", figsize=(15,10))

# If we want to exclude the Antarctic region
ax.set_xlim(-180,180)
ax.set_ylim(-65, 90)


plt.show()



In [ ]:
### spatial join : if POINT is inside REGION -> join 
# per person we add the country of birth
pgr_reg_countries = pgr_reg.sjoin(countries)
pgr_reg_countries.iloc[[1,5,10]]

In [ ]:
### Liyt of column names
print(pgr_reg_countries.columns)

In [ ]:
### drop not needed columns
pgr_reg_countries = pgr_reg_countries.drop([ 'index_right', 'NAME_FREN', 'SVRG_UN', 'CAPT',
       'EU_STAT', 'EFTA_STAT', 'CC_STAT', 'NAME_GERM'], axis=1)
pgr_reg_countries.head(2)

### Clean up gender variable

In [ ]:
print(pgr_reg_countries.groupby(by='gender').size())

In [ ]:
### Limit to female/male for statistic treatment
pgr_reg_countries=pgr_reg_countries[pgr_reg_countries.gender.isin(['female', 'male'])]

In [ ]:
print(pgr_reg_countries.groupby(by='gender').size())

### Prepare the data for the bivariate analysis, cf. challenge da3

In [ ]:
file_address='da_data/da3-birthYear-gender-region-country.csv'
pgr_reg_countries.to_csv(file_address, index=False)

In [ ]:
### Create columns for long / lat
pgr_reg_countries['long'] = pgr_reg_countries['geometry'].x
pgr_reg_countries['lat'] = pgr_reg_countries['geometry'].y
pgr_reg_countries.head()

### Group by regions and periods

In [ ]:
reg_per=pgr_reg.groupby(['FID', 'REGION', 'periods'], observed=True, as_index=False).size()
reg_per.head(3)

In [ ]:
### Merge with world to get polygons
geom_reg_per=pd.merge(reg_per, world, left_on='FID', right_on='FID')
geom_reg_per=geom_reg_per.drop(['REGION_y','SQMI','SQKM'], axis=1)
geom_reg_per.columns=['FID', 'name','periods','number','geometry']
geom_reg_per.head(2)

In [ ]:
### Transform to geoDataFrame
grp_geo = gpd.GeoDataFrame(
    geom_reg_per, 
    crs=4326
)
grp_geo.head(2)

In [ ]:
print(len(grp_geo))

In [ ]:
### Tableau de contingence
X = "REGION"
Y = "periods"  # "0"

pv_per_reg = pgr_reg[[X,Y]].pivot_table(index=Y,columns=X,observed=True, aggfunc=len,margins=True,margins_name="Total").fillna(0).astype(int)
pv_per_reg

In [ ]:
### Table translation
df_t = pv_per_reg.T

In [ ]:
df_t[df_t.Total > 200]

In [ ]:
print(list(df_t.index))

In [ ]:
grp_geo.head(1)

In [ ]:
sel_geo = grp_geo[grp_geo.name.isin(list(df_t.index))].copy(deep=True)
sel_geo.head(2)

In [ ]:
print(grp_geo.crs)

In [ ]:
grp_geo=grp_geo.sort_values(by='periods')

In [ ]:
grp_geo = gpd.GeoDataFrame(
    geom_reg_per, 
    crs=4326
)

### simplify geometry 
## https://www.statology.org/how-to-simplify-geographic-data-using-geopandas/
grp_geo["geometry"] = (
    grp_geo.simplify(tolerance=0.05)
)

In [ ]:
import os, json
import plotly.express as px

OUTPUT_DIR  = "interactive_images/birth_periods_choropleth_frames"
OUTPUT_MAIN = "interactive_images/birth_periods_regions_choropleth.html"
os.makedirs(OUTPUT_DIR, exist_ok=True)

max_val  = float(grp_geo["number"].max())
periods  = sorted(grp_geo["periods"].unique())
period_strs = [str(p) for p in periods]

# ── 1. GeoJSON once (all features, compact JSON) ──────────────────────────────
geojson = grp_geo.__geo_interface__
with open(f"{OUTPUT_DIR}/geojson.json", "w") as f:
    json.dump(geojson, f, separators=(",", ":"))

# ── 2. Per-period data (tiny: just index + values + hover labels) ─────────────
period_file_map = {}
for period in periods:
    df_p = grp_geo[grp_geo["periods"] == period]
    safe = str(period).replace("/", "-").replace(" ", "_")
    fname = f"{safe}.json"
    payload = {
        "locations": df_p.index.tolist(),
        "z":         df_p["number"].tolist(),
        "text":      df_p["name"].tolist(),
    }
    with open(f"{OUTPUT_DIR}/{fname}", "w") as f:
        json.dump(payload, f, separators=(",", ":"))
    period_file_map[str(period)] = fname

print(f"Saved {len(periods)} period files to {OUTPUT_DIR}/")

# ── 3. Main HTML (Plotly from CDN, slider + play/pause) ──────────────────────
FRAME_DIR  = "images/birth_periods_choropleth_frames/"   # relative to the HTML
FRAME_DUR  = 2000   # ms per frame during auto-play

main_html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Birth regions by period</title>
  <script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>
  <link rel="stylesheet" href="styles.css">
  <style>
    body  {{ font-family: sans-serif; margin: 0; padding: 10px; background: #fff; }}
    #ctrl {{ display: flex; align-items: center; gap: 12px; margin-bottom: 8px; flex-wrap: wrap; }}
    #period-label {{ font-weight: 600; min-width: 120px; }}
    #play-btn {{
      background: #1f77b4; color: #fff; border: none;
      padding: 6px 16px; border-radius: 4px; cursor: pointer; font-size: 14px;
    }}
    #period-slider {{ flex: 1; min-width: 200px; max-width: 600px; }}
    #map {{ width: 1000px; max-width: 100%; height: 800px; }}
    #status {{ font-size: 12px; color: #888; }}
  </style>
</head>
<body>
 <div id="map"></div>
  <div id="ctrl">
    <button id="play-btn" onclick="togglePlay()">&#9654; Play</button>
    <input  id="period-slider" type="range"
            min="0" max="{len(periods) - 1}" value="0"
            oninput="onSlider(this.value)">
    <span   id="period-label">{period_strs[0]}</span>
  </div>
 
  <p id="status">Loading…</p>

  <script>
    const periods   = {json.dumps(period_strs)};
    const fileMap   = {json.dumps(period_file_map)};
    const frameDir  = "{FRAME_DIR}";
    const maxVal    = {max_val};
    const frameDur  = {FRAME_DUR};

    let geojson   = null;
    let curIdx    = 0;
    let playing   = false;
    let playTimer = null;
    let plotted   = false;
    const cache   = {{}};          // period string → data payload

    // ── fetch helpers ────────────────────────────────────────────────────────
    async function fetchJSON(url) {{
      const r = await fetch(url);
      if (!r.ok) throw new Error(`HTTP ${{r.status}} ${{url}}`);
      return r.json();
    }}

    async function getPeriodData(period) {{
      if (!cache[period]) cache[period] = await fetchJSON(frameDir + fileMap[period]);
      return cache[period]}};
    

    // ── render ───────────────────────────────────────────────────────────────
    async function loadPeriod(period) {{
      document.getElementById("period-label").textContent = period;
      document.getElementById("period-slider").value = periods.indexOf(period);
      document.getElementById("status").textContent = `Loading ${{period}}…`;

      const d = await getPeriodData(period);
      document.getElementById("status").textContent = "";

      const trace = {{
        type:               "choroplethmapbox",
        geojson:            geojson,
        locations:          d.locations,
        z:                  d.z,
        text:               d.text,
        hovertemplate:      "%{{text}}: %{{z}}<extra></extra>",
        colorscale:         "Blues",
        reversescale:   true, 
        zmin:               0,
        zmax:               maxVal,
        colorbar:           {{ title: "Count" }},
        marker:             {{ opacity: 0.8, line: {{ width: 0.5, color: "#fff" }} }},
      }};

      const layout = {{
        mapbox:  {{ style: "carto-positron", center: {{ lat: 47, lon: 8.5 }}, zoom: 1 }},
        margin:  {{ l: 0, r: 20, t: 30, b: 20 }},
        height:  800,
        width:   1000,
        title:   {{ text: `Birth regions — ${{period}}`, x: 0.5 }},
      }};

      if (!plotted) {{
        Plotly.newPlot("map", [trace], layout, {{responsive: true}});
        plotted = true;
      }} else {{
        Plotly.react("map", [trace], layout);
      }}
    }}

    // ── controls ─────────────────────────────────────────────────────────────
    function onSlider(val) {{
      curIdx = +val;
      loadPeriod(periods[curIdx]);
    }}

    function togglePlay() {{
      playing = !playing;
      document.getElementById("play-btn").textContent = playing ? "⏸ Pause" : "▶ Play";
      if (playing) {{
        advance();
        playTimer = setInterval(advance, frameDur);
      }} else {{
        clearInterval(playTimer);
      }}
    }}

    function advance() {{
      curIdx = (curIdx + 1) % periods.length;
      loadPeriod(periods[curIdx]);
      if (curIdx === periods.length - 1 && playing) {{ togglePlay(); }}
    }}

    // ── init ─────────────────────────────────────────────────────────────────
    (async () => {{
      geojson = await fetchJSON(frameDir + "geojson.json");
      // pre-fetch first two periods for snappy start
      await loadPeriod(periods[0]);
      if (periods[1]) getPeriodData(periods[1]);
    }})();
  </script>
</body>
</html>"""

with open(OUTPUT_MAIN, "w", encoding="utf-8") as f:
    f.write(main_html)

print(f"Main file: {OUTPUT_MAIN}")

### [under construction] Group by regions and periods: only female persons

In [ ]:
reg_per_f=pgr_reg[pgr_reg.gender=='female'].groupby(['FID', 'REGION', 'periods'], observed=True, as_index=False).size()
reg_per_f.head(3)

In [ ]:
### Merge with world to get polygons
geom_reg_per_f=pd.merge(reg_per_f, world, left_on='FID', right_on='FID')
geom_reg_per_f=geom_reg_per.drop(['REGION_y','SQMI','SQKM'], axis=1)
geom_reg_per_f.columns=['FID', 'name','periods','number','geometry']
geom_reg_per_f.head(2)

In [ ]:
### Transform to geoDataFrame
grp_geo = gpd.GeoDataFrame(
    geom_reg_per, 
    crs=4326
)
grp_geo.head(2)

In [ ]:
print(len(grp_geo))

In [ ]:
### Tableau de contingence
X = "REGION"
Y = "periods"  # "0"

pv_per_reg = pgr_reg[[X,Y]].pivot_table(index=Y,columns=X,observed=True, aggfunc=len,margins=True,margins_name="Total").fillna(0).astype(int)
# pv_per_reg

In [ ]:
### Table translation
df_t = pv_per_reg.T

In [ ]:
# df_t[df_t.Total > 200]

In [ ]:
print(list(df_t.index))

In [ ]:
grp_geo.head(1)

In [ ]:
sel_geo = grp_geo[grp_geo.name.isin(list(df_t.index))].copy(deep=True)
sel_geo.head(2)

In [ ]:
print(grp_geo.crs)

In [ ]:
grp_geo = gpd.GeoDataFrame(
    geom_reg_per, 
    crs=4326
)

### simplify geometry 
## https://www.statology.org/how-to-simplify-geographic-data-using-geopandas/
grp_geo["geometry"] = (
    grp_geo.simplify(tolerance=0.05)
)

In [ ]:
grp_geo=grp_geo.sort_values(by='periods')

In [ ]:
max = grp_geo.number.max()

# Create the choropleth map
fig = px.choropleth_map(
    grp_geo,
    geojson=grp_geo.__geo_interface__,
    locations=grp_geo.index,
    color='number',  # Replace with your actual column name
    color_continuous_scale='RdPu',
    range_color=(0, max),  # Replace with your actual range
    zoom=1,
    #map_style='white-bg',
    hover_name='name',
    center={'lat': 47, 'lon': 8.5},
    height=800,
    width=1000,
    animation_frame="periods",
    #title="Temporal evolution of birth regions"

)


fig.update_layout(
    #mapbox={"style": "carto-positron", "zoom":2},
    margin={"l": 0, "r": 20, "t": 30, "b": 200}
)



fig.layout.updatemenus[0].buttons[0].args[1]['frame']['duration'] = 2000
fig.layout.updatemenus[0].buttons[0].args[1]['transition']['duration'] = 20



f_address = "interactive_images/birth_periods_regions_choropleth_female.html"
fig.write_html(f_address)
# Show the plot
# fig.show()